# TimesFM vs classical baselines — results dashboard

This notebook **reads the CSVs** written by the experiment notebooks in `output/` and shows:

- one **combined table** (all scenarios),
- **per-experiment** tables when useful,
- a **bar chart** of MSE (lower is better),
- a **Δ MSE** column: `TimesFM − baseline` ( **negative** ⇒ TimesFM wins on MSE ).

**Expected files** (run the other notebooks first if a file is missing):
`ar_vs_timesfm_results.csv`, `ma_vs_timesfm_results.csv`, `arma_vs_timesfm_results.csv`, `seasonal_airline_vs_timesfm_results.csv`, `hamilton_switching_vs_timesfm_results.csv`.

## 1. Paths and imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError("Install matplotlib for charts: pip install matplotlib") from e

HERE = Path.cwd().resolve()
if (HERE / "output").is_dir():
    OUT = HERE / "output"
elif (HERE / "experiments_nikhilesh_experiments" / "output").is_dir():
    OUT = HERE / "experiments_nikhilesh_experiments" / "output"
else:
    OUT = HERE / "output"

## 2. Load and normalize each CSV into one long table

In [ ]:
def load_if(path: Path) -> pd.DataFrame | None:
    if not path.is_file():
        return None
    return pd.read_csv(path)


pieces: list[pd.DataFrame] = []

df = load_if(OUT / "ar_vs_timesfm_results.csv")
if df is not None:
    for _, r in df.iterrows():
        pieces.append(
            {
                "Experiment": "AR",
                "Scenario": f"p = {int(r['p_dgp'])}",
                "Baseline": "ARIMA(p,0,0)",
                "MSE baseline": r["mse_ar"],
                "MSE TimesFM": r["mse_timesfm"],
            }
        )

df = load_if(OUT / "ma_vs_timesfm_results.csv")
if df is not None:
    for _, r in df.iterrows():
        pieces.append(
            {
                "Experiment": "MA",
                "Scenario": f"q = {int(r['q_dgp'])}",
                "Baseline": "ARIMA(0,0,q)",
                "MSE baseline": r["mse_ma"],
                "MSE TimesFM": r["mse_timesfm"],
            }
        )

df = load_if(OUT / "arma_vs_timesfm_results.csv")
if df is not None:
    for _, r in df.iterrows():
        pieces.append(
            {
                "Experiment": "ARMA",
                "Scenario": f"(p,q) = ({int(r['p_dgp'])},{int(r['q_dgp'])})",
                "Baseline": "ARIMA(p,0,q)",
                "MSE baseline": r["mse_arma"],
                "MSE TimesFM": r["mse_timesfm"],
            }
        )

df = load_if(OUT / "seasonal_airline_vs_timesfm_results.csv")
if df is not None:
    for _, r in df.iterrows():
        pieces.append(
            {
                "Experiment": "Seasonal (airline)",
                "Scenario": f"m={int(r['seasonal_period'])}, k_first={int(r['k_first'])}",
                "Baseline": "SARIMA(0,1,1)(0,1,1)ₘ",
                "MSE baseline": r["mse_airline_sarimax"],
                "MSE TimesFM": r["mse_timesfm"],
            }
        )

df = load_if(OUT / "hamilton_switching_vs_timesfm_results.csv")
if df is not None:
    for _, r in df.iterrows():
        pieces.append(
            {
                "Experiment": "Markov switching",
                "Scenario": "MS-AR(1) DGP",
                "Baseline": "OLS AR(1)",
                "MSE baseline": r["mse_ols_ar1"],
                "MSE TimesFM": r["mse_timesfm"],
            }
        )

if not pieces:
    raise FileNotFoundError(
        f"No result CSVs found under {OUT}. Run the experiment notebooks first."
    )

summary = pd.DataFrame(pieces)
summary["Δ (TF − baseline)"] = summary["MSE TimesFM"] - summary["MSE baseline"]
summary["TimesFM better (MSE)"] = summary["Δ (TF − baseline)"] < 0
summary

## 3. Styled table (sort by experiment, then scenario)

In [ ]:
display_df = summary.sort_values(["Experiment", "Scenario"]).reset_index(drop=True)
fmt = {c: "{:.4f}" for c in ["MSE baseline", "MSE TimesFM", "Δ (TF − baseline)"]}
styled = (
    display_df.style.format(fmt)
    .background_gradient(subset=["MSE baseline", "MSE TimesFM"], cmap="YlOrRd_r")
    .set_properties(**{"text-align": "right"}, subset=["MSE baseline", "MSE TimesFM", "Δ (TF − baseline)"])
)
styled

## 4. Chart — MSE by scenario

Each **label** is one row of the table; **two bars** per label (baseline vs TimesFM).

In [ ]:
plot_df = display_df.copy()
plot_df["Label"] = plot_df["Experiment"] + " | " + plot_df["Scenario"]
n = len(plot_df)
fig_h = max(4.0, 0.35 * n)
fig, ax = plt.subplots(figsize=(10, fig_h))
y = np.arange(n)
h = 0.35
ax.barh(y - h / 2, plot_df["MSE baseline"], height=h, label="Baseline", color="#2c5282")
ax.barh(y + h / 2, plot_df["MSE TimesFM"], height=h, label="TimesFM 2.5", color="#c05621")
ax.set_yticks(y)
ax.set_yticklabels(plot_df["Label"], fontsize=9)
ax.set_xlabel("MSE (lower is better)")
ax.legend(loc="lower right")
ax.set_title("One-step MSE: classical baseline vs TimesFM")
ax.grid(axis="x", alpha=0.35)
plt.tight_layout()
plt.show()

## 5. Optional — export combined CSV

Writes `output/all_experiments_summary.csv` for slides or papers.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
save_path = OUT / "all_experiments_summary.csv"
display_df.to_csv(save_path, index=False)
print(save_path)